# COVID-19 Case Forecasting & Visualization
End-to-end notebook companion to the modular project.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, BayesianRidge


## Load processed data

In [ ]:
df = pd.read_csv('../data/processed/covid19_global_processed.csv', parse_dates=['date'])
df.tail()


## Exploratory visualization

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df.date, df.confirmed, label='Confirmed')
plt.plot(df.date, df.deaths, label='Deaths')
plt.plot(df.date, df.recovered, label='Recovered')
plt.legend(); plt.title('Global COVID-19 Trends'); plt.show()


## Chronological train/test split

In [ ]:
work = df.tail(min(180, len(df))).dropna(subset=['days_since_start'])
X = work[['days_since_start']].values
y = work['confirmed'].values
split = int(len(work) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


## Train three models

In [ ]:
models = {
    'SVR': Pipeline([('scale', StandardScaler()), ('model', SVR(kernel='poly', degree=3, C=0.1, gamma=0.01, epsilon=1))]),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(3, include_bias=False)), ('model', LinearRegression())]),
    'Bayesian Ridge': Pipeline([('poly', PolynomialFeatures(3, include_bias=False)), ('model', BayesianRidge())]),
}
metrics = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics.append({'Model': name, 'MAE': mean_absolute_error(y_test, pred), 'MSE': mean_squared_error(y_test, pred)})
pd.DataFrame(metrics)


## 10-day forecast

In [ ]:
future = pd.DataFrame({'days_since_start': range(int(df.days_since_start.max())+1, int(df.days_since_start.max())+11)})
for name, model in models.items():
    future[name] = model.predict(future[['days_since_start']])
future['date'] = pd.date_range(df.date.max() + pd.Timedelta(days=1), periods=10)
future
